In [ ]:
"""
U-AutoRec for MovieLens 1M  —  Corrected Implementation
=========================================================
Paper: "AutoRec: Autoencoders Meet Collaborative Filtering"
Sedhain et al., WWW 2015   |   Target RMSE: 0.874

ROOT CAUSES of 0.978 RMSE fixed here:
  FIX 1 — L2 reg was computed per-batch and added to per-sample loss.
           Correct: global L2 on full weight matrices, added ONCE per
           forward pass at the batch level, not scaled by batch size.

  FIX 2 — Rating matrix was built from FULL data instead of train split.
           Correct: matrix built only from train_data per fold.

  FIX 3 — RMSE evaluation was computing average of per-user RMSEs.
           Correct: collect all (pred, target) pairs then one global RMSE.

  FIX 4 — build_rating_matrix used a Python loop (very slow).
           Correct: vectorised numpy indexing — 100x faster.

  FIX 5 — Only best model per fold saved (periodic saves removed).

Kaggle-ready:
  - No argparse (Config dataclass — no Jupyter/kernel conflict)
  - num_workers=0  (Kaggle forbids DataLoader forking)
  - Auto-discovers ratings.dat under the dataset root folder
  - All writes go to /kaggle/working/
"""

import os
import time
import random
import dataclasses as _dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass
import logging

# ──────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────
# Config  ← only thing you need to edit
# ──────────────────────────────────────────────────────────────
@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────
    # Folder containing your ML-1M data (ratings.dat found automatically)
    dataset_root:   str   = "/kaggle/input/datasets/priyanshuunayak/ratingsdata"
    checkpoint_dir: str   = "/kaggle/working/u_autorec_1m"

    # ── Model  (paper best) ───────────────────────────────────
    hidden_units:   int   = 500       # paper Figure 2: performance peaks at 500

    # ── Training  (paper exact settings) ─────────────────────
    epochs:         int   = 500       # paper trains up to convergence
    batch_size:     int   = 256       # users per mini-batch
    lr:             float = 0.001     # RProp initial step size
    # L2 reg: paper tunes over {0.001, 0.01, 0.1, 1, 100, 1000}
    # U-AutoRec on ML-1M best value is 0.001 (much lower than I-AutoRec)
    lambda_reg:     float = 0.001

    # ── Early stopping ────────────────────────────────────────
    early_stop:     int   = 40        # patience epochs on val RMSE
    log_every:      int   = 10        # print every N epochs

    # ── System ────────────────────────────────────────────────
    seed:           int   = 42
    num_workers:    int   = 0         # must be 0 on Kaggle
    device:         str   = "auto"    # "auto" | "cuda" | "cpu"


# ──────────────────────────────────────────────────────────────
# Utilities
# ──────────────────────────────────────────────────────────────
def cfg_to_dict(cfg):
    try:
        return _dc.asdict(cfg)
    except TypeError:
        return vars(cfg)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False


# ──────────────────────────────────────────────────────────────
# Auto-discover ratings.dat
# ──────────────────────────────────────────────────────────────
def find_ratings_file(root: str) -> str:
    logger.info(f"Searching for ratings.dat under: {root}")
    for dirpath, _, files in os.walk(root):
        if "ratings.dat" in files:
            path = os.path.join(dirpath, "ratings.dat")
            logger.info(f"  Found: {path}")
            return path

    all_files = [
        os.path.join(d, f)
        for d, _, fs in os.walk(root)
        for f in fs
    ]
    listing = "\n  ".join(all_files) or "(empty)"
    raise FileNotFoundError(
        f"ratings.dat not found under '{root}'.\n"
        f"Files present:\n  {listing}\n"
        f"→ Fix 'dataset_root' in Config."
    )


# ──────────────────────────────────────────────────────────────
# Data Loading & Splitting
# ──────────────────────────────────────────────────────────────
def load_ml1m(path: str):
    """
    Parse ratings.dat → zero-indexed numpy array (n_ratings, 3).
    Columns: [user_idx, item_idx, rating]
    """
    logger.info(f"Loading: {path}")
    raw = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("::")
            if len(parts) < 3:
                continue
            raw.append((int(parts[0]), int(parts[1]), float(parts[2])))

    raw = np.array(raw, dtype=np.float64)   # (N, 3)

    uids = raw[:, 0].astype(int)
    iids = raw[:, 1].astype(int)

    u2i = {u: i for i, u in enumerate(sorted(set(uids)))}
    i2i = {it: i for i, it in enumerate(sorted(set(iids)))}

    data = np.column_stack([
        [u2i[u] for u in uids],
        [i2i[i] for i in iids],
        raw[:, 2],
    ]).astype(np.float64)

    n_users = len(u2i)
    n_items = len(i2i)
    logger.info(f"  Users={n_users:,}  Items={n_items:,}  Ratings={len(data):,}")
    return data, n_users, n_items


def split_90_10(data, seed):
    """90% train+val / 10% test — paper exact procedure."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data))
    cut = int(len(data) * 0.9)
    return data[idx[:cut]], data[idx[cut:]]


def split_val(trainval, seed):
    """10% of trainval → validation."""
    rng = np.random.default_rng(seed + 1000)   # different seed from test split
    idx = rng.permutation(len(trainval))
    cut = int(len(trainval) * 0.9)
    return trainval[idx[:cut]], trainval[idx[cut:]]


# ──────────────────────────────────────────────────────────────
# Rating Matrix  (FIX 4: vectorised, not Python loop)
# ──────────────────────────────────────────────────────────────
def make_rating_matrix(data, n_users, n_items):
    """
    Build dense (n_users × n_items) matrix from (user, item, rating) triples.
    Unobserved entries = 0.0  →  used as mask.
    IMPORTANT: built from TRAIN data only, never from full dataset.
    """
    R = np.zeros((n_users, n_items), dtype=np.float32)
    rows = data[:, 0].astype(int)
    cols = data[:, 1].astype(int)
    vals = data[:, 2].astype(np.float32)
    R[rows, cols] = vals
    return R


# ──────────────────────────────────────────────────────────────
# Dataset
# ──────────────────────────────────────────────────────────────
class UserRatingDataset(Dataset):
    """
    One sample = one user's partially observed item rating vector.
    Returns (rating_vec, mask_vec) where mask[j]=1 iff user rated item j.
    """
    def __init__(self, R: np.ndarray):
        self.R    = torch.from_numpy(R).float()          # (n_users, n_items)
        self.mask = (self.R != 0).float()                # (n_users, n_items)

    def __len__(self):
        return self.R.shape[0]

    def __getitem__(self, idx):
        return self.R[idx], self.mask[idx]


# ──────────────────────────────────────────────────────────────
# Model
# ──────────────────────────────────────────────────────────────
class UAutoRec(nn.Module):
    """
    User-based AutoRec.

        h(r; θ) = f( W · g(V·r + μ) + b )

    g(·) = Sigmoid   ← hidden activation  (paper Table 1b best)
    f(·) = Identity  ← output activation  (paper Table 1b best)

    Regularisation on ||W||²_F + ||V||²_F  (not biases).
    """
    def __init__(self, n_items: int, k: int = 500):
        super().__init__()
        self.encoder = nn.Linear(n_items, k)      # V, μ
        self.decoder = nn.Linear(k, n_items)      # W, b
        self.g = nn.Sigmoid()
        # f = Identity → no module needed
        self._init()

    def _init(self):
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, r):
        return self.decoder(self.g(self.encoder(r)))

    def l2_penalty(self):
        """Global L2 on weight matrices only — computed once per step."""
        return (
            self.encoder.weight.norm(p="fro") ** 2 +
            self.decoder.weight.norm(p="fro") ** 2
        )


# ──────────────────────────────────────────────────────────────
# Loss  (FIX 1: correct masked MSE — observed ratings only)
# ──────────────────────────────────────────────────────────────
def masked_mse(pred, target, mask):
    """
    MSE computed ONLY over observed ratings (mask == 1).
    This matches  ||r - h(r)||²_O  in paper Eq.2.

    NOTE: we do NOT divide by batch_size here — the L2 term is
    added at the batch level, so both terms scale identically.
    """
    diff = (pred - target) * mask
    n_obs = mask.sum()
    if n_obs == 0:
        return torch.tensor(0.0, requires_grad=True)
    return (diff ** 2).sum() / n_obs


# ──────────────────────────────────────────────────────────────
# Evaluation  (FIX 3: global RMSE over all test pairs)
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_rmse(model, pairs, R_train, device, chunk=1024):
    """
    Compute RMSE on (user, item, rating) pairs using the trained model.

    pairs    : numpy array (N, 3)  — (user_idx, item_idx, true_rating)
    R_train  : (n_users, n_items) training rating matrix (used as model input)
    chunk    : users processed per GPU batch to avoid OOM

    Predictions are clipped to [1, 5] (valid ML-1M rating range).
    """
    model.eval()
    n_users = R_train.shape[0]

    # Reconstruct entire user-item matrix in chunks
    recon = np.empty_like(R_train)
    for s in range(0, n_users, chunk):
        batch = torch.from_numpy(R_train[s:s+chunk]).float().to(device)
        recon[s:s+chunk] = model(batch).cpu().numpy()

    # Gather predictions for test pairs
    u_idx = pairs[:, 0].astype(int)
    i_idx = pairs[:, 1].astype(int)
    preds   = np.clip(recon[u_idx, i_idx], 1.0, 5.0)
    targets = pairs[:, 2]

    return float(np.sqrt(np.mean((preds - targets) ** 2)))


# ──────────────────────────────────────────────────────────────
# One Training Epoch
# ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, lambda_reg, device):
    """
    One pass over all users.

    Loss per step = masked_MSE(pred, R, mask)
                  + (lambda / 2) * (||V||²_F + ||W||²_F)

    RProp updates step sizes per-parameter — no LR scheduling needed.
    """
    model.train()
    running_loss = 0.0

    bar = tqdm(loader, desc="  train", leave=False, ncols=88)
    for R_batch, mask_batch in bar:
        R_batch    = R_batch.to(device, non_blocking=True)
        mask_batch = mask_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(R_batch)

        # Reconstruction loss — observed entries only
        rec_loss = masked_mse(pred, R_batch, mask_batch)

        # Global L2 regularisation on weight matrices
        # (FIX 1: model.l2_penalty() is global, not per-sample)
        l2_loss  = (lambda_reg / 2.0) * model.l2_penalty()

        loss = rec_loss + l2_loss
        loss.backward()
        optimizer.step()

        running_loss += rec_loss.item()   # track recon loss only for monitoring
        bar.set_postfix(rec=f"{rec_loss.item():.4f}")

    return running_loss / len(loader)


# ──────────────────────────────────────────────────────────────
# Single Fold Training
# ──────────────────────────────────────────────────────────────
def run_fold(fold_id, train_data, val_data, test_data,
             n_users, n_items, cfg, device):

    logger.info(f"\n{'='*62}")
    logger.info(f"  FOLD {fold_id}/5   "
                f"train={len(train_data):,}  "
                f"val={len(val_data):,}  "
                f"test={len(test_data):,}")
    logger.info(f"{'='*62}")

    # ── Build train rating matrix from train_data ONLY  (FIX 2) ──
    R_train = make_rating_matrix(train_data, n_users, n_items)

    dataset = UserRatingDataset(R_train)
    loader  = DataLoader(
        dataset,
        batch_size  = cfg.batch_size,
        shuffle     = True,
        num_workers = cfg.num_workers,
        pin_memory  = (device.type == "cuda"),
        drop_last   = False,
    )

    model = UAutoRec(n_items=n_items, k=cfg.hidden_units).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"  Model: UAutoRec  k={cfg.hidden_units}  params={n_params:,}")

    # RProp — exact optimiser from paper
    optimizer = optim.Rprop(
        model.parameters(),
        lr         = cfg.lr,
        etas       = (0.5, 1.2),
        step_sizes = (1e-6, 50),
    )

    best_dir       = cfg.checkpoint_dir
    best_path      = os.path.join(best_dir, f"fold{fold_id}_best.pt")
    best_val_rmse  = float("inf")
    best_epoch     = 0
    patience       = 0

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()

        train_loss = train_epoch(model, loader, optimizer, cfg.lambda_reg, device)
        val_rmse   = evaluate_rmse(model, val_data,  R_train, device)
        elapsed    = time.time() - t0

        if epoch % cfg.log_every == 0 or epoch == 1:
            logger.info(
                f"  Ep {epoch:4d}/{cfg.epochs}  "
                f"loss={train_loss:.5f}  "
                f"val_RMSE={val_rmse:.4f}  "
                f"best={best_val_rmse:.4f}  "
                f"({elapsed:.1f}s)"
            )

        # ── Save best model only  (periodic saves removed per request) ──
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch    = epoch
            patience      = 0
            torch.save({
                "fold":       fold_id,
                "epoch":      epoch,
                "model":      model.state_dict(),
                "val_rmse":   val_rmse,
                "config":     cfg_to_dict(cfg),
            }, best_path)
            logger.info(f"  ✓ New best  val_RMSE={val_rmse:.4f}  → {os.path.basename(best_path)}")
        else:
            patience += 1

        if cfg.early_stop > 0 and patience >= cfg.early_stop:
            logger.info(f"  Early stop at epoch {epoch} (patience={cfg.early_stop})")
            break

    # ── Final test evaluation with best model ──
    ckpt = torch.load(best_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model"])
    test_rmse = evaluate_rmse(model, test_data, R_train, device)

    logger.info(f"\n  Fold {fold_id} done  →  "
                f"best val={best_val_rmse:.4f} (ep {best_epoch})  "
                f"test_RMSE={test_rmse:.4f}")

    return test_rmse


# ──────────────────────────────────────────────────────────────
# 5-Fold Cross Validation
# ──────────────────────────────────────────────────────────────
def run_cv(all_data, n_users, n_items, cfg, device):
    """
    Paper procedure (Section 3):
      - Randomly split into 90% train / 10% test, repeated 5 times
      - Hold out 10% of train as validation (for early stopping)
      - Report mean ± std RMSE over 5 folds
    """
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    logger.info("\n" + "="*62)
    logger.info("  5-Fold Cross Validation  (paper procedure)")
    logger.info("="*62)

    fold_rmses = []

    for fold in range(1, 6):
        set_seed(cfg.seed + fold)

        trainval, test  = split_90_10(all_data,  seed=cfg.seed + fold)
        train,    val   = split_val(trainval,    seed=cfg.seed + fold)

        rmse = run_fold(
            fold_id    = fold,
            train_data = train,
            val_data   = val,
            test_data  = test,
            n_users    = n_users,
            n_items    = n_items,
            cfg        = cfg,
            device     = device,
        )
        fold_rmses.append(rmse)

    avg = float(np.mean(fold_rmses))
    std = float(np.std(fold_rmses))

    logger.info("\n" + "="*62)
    logger.info("  RESULTS")
    logger.info("="*62)
    for i, r in enumerate(fold_rmses):
        logger.info(f"  Fold {i+1}:  RMSE = {r:.4f}")
    logger.info(f"\n  Mean RMSE : {avg:.4f}  ±  {std:.4f}")
    logger.info(f"  Paper     : 0.874  (U-AutoRec ML-1M)")
    logger.info("="*62)

    # Save summary
    out = os.path.join(cfg.checkpoint_dir, "results.txt")
    with open(out, "w") as f:
        f.write("U-AutoRec ML-1M  —  Replication\n")
        f.write("="*40 + "\n\n")
        for i, r in enumerate(fold_rmses):
            f.write(f"Fold {i+1}: {r:.4f}\n")
        f.write(f"\nMean : {avg:.4f} ± {std:.4f}\n")
        f.write(f"Paper: 0.874\n\n")
        f.write(f"Config: {cfg_to_dict(cfg)}\n")
    logger.info(f"Summary → {out}")

    return avg, std, fold_rmses


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main():
    cfg = Config()
    set_seed(cfg.seed)

    # Device
    if cfg.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(cfg.device)

    logger.info(f"Device : {device}")
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        logger.info(f"  GPU  : {props.name}")
        logger.info(f"  VRAM : {props.total_memory / 1e9:.1f} GB")

    # Locate data
    ratings_path = find_ratings_file(cfg.dataset_root)

    # Load
    all_data, n_users, n_items = load_ml1m(ratings_path)

    # Config summary
    logger.info(f"\n--- Config ---")
    logger.info(f"  hidden_units : {cfg.hidden_units}")
    logger.info(f"  lambda_reg   : {cfg.lambda_reg}  ← key param (0.001 for U-AutoRec)")
    logger.info(f"  epochs       : {cfg.epochs}")
    logger.info(f"  batch_size   : {cfg.batch_size}")
    logger.info(f"  early_stop   : {cfg.early_stop}")
    logger.info(f"  optimizer    : RProp (lr={cfg.lr})")
    logger.info(f"  activation   : g=Sigmoid, f=Identity  (Table 1b best)")
    logger.info(f"  num_workers  : {cfg.num_workers}")

    run_cv(all_data, n_users, n_items, cfg, device)


if __name__ == "__main__":
    main()

21:12:02 | INFO | Device : cuda
21:12:02 | INFO |   GPU  : Tesla T4
21:12:02 | INFO |   VRAM : 15.6 GB
21:12:02 | INFO | Searching for ratings.dat under: /kaggle/input/datasets/priyanshuunayak/ratingsdata
21:12:02 | INFO |   Found: /kaggle/input/datasets/priyanshuunayak/ratingsdata/ratings.dat
21:12:02 | INFO | Loading: /kaggle/input/datasets/priyanshuunayak/ratingsdata/ratings.dat
21:12:04 | INFO |   Users=6,040  Items=3,706  Ratings=1,000,209
21:12:04 | INFO | 
--- Config ---
21:12:04 | INFO |   hidden_units : 500
21:12:04 | INFO |   lambda_reg   : 0.001  ← key param (0.001 for U-AutoRec)
21:12:04 | INFO |   epochs       : 500
21:12:04 | INFO |   batch_size   : 256
21:12:04 | INFO |   early_stop   : 40
21:12:04 | INFO |   optimizer    : RProp (lr=0.001)
21:12:04 | INFO |   activation   : g=Sigmoid, f=Identity  (Table 1b best)
21:12:04 | INFO |   num_workers  : 0
21:12:04 | INFO | 
21:12:04 | INFO |   5-Fold Cross Validation  (paper procedure)
21:12:04 | INFO | =======================